# Family-aware TF-IDF diagnostic

This diagnostic compares direct 18-class classification against a hierarchical provider-family classifier on the same grouped folds. It uses only training-fold information during evaluation and does not use test data or prompt-relative features.


In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score

DATA_DIR=Path('/kaggle/input/competitions/name-that-llm')
OUT=Path('/kaggle/working/family_aware_results.json')
SEED=42; N_SPLITS=5; C=2.0
FAMILY={
'gpt-5.5':'openai','gpt-5.6-luna':'openai','gpt-5.6-sol':'openai','gpt-5.6-terra':'openai','gpt-6-astra':'openai',
'gemini-3.1-pro-preview':'google','gemini-3.6-flash':'google','gemini-3.7-flash':'google','gemini-3.8-flash':'google',
'claude-fable-5':'anthropic','claude-opus-4-6':'anthropic','claude-opus-4-8':'anthropic','claude-opus-5':'anthropic','claude-sonnet-5':'anthropic',
'qwen3.8-max':'qwen','glm-5.3':'zai','glm-5.3-flash':'zai','kimi-k3':'moonshot'}
train=pd.read_csv(DATA_DIR/'train.csv'); assert {'text','model'}.issubset(train.columns)
text=train.text.fillna('').astype(str).to_numpy(); y=train.model.astype(str).to_numpy(); family=np.array([FAMILY[v] for v in y])
groups=train.prompt_id.astype(str).to_numpy() if 'prompt_id' in train else None
PUNCT=tuple(".,;:!?()[]{}\"'/-")
def style(items):
 rows=[]
 for s in items:
  words=re.findall(r"\b[\w']+\b",s); den=max(len(s),1)
  rows.append([len(s),len(words),len(re.findall(r'[.!?]+',s)),sum(s.count(c) for c in PUNCT),sum(c.isdigit() for c in s),sum(c.isupper() for c in s),s.count('?'),s.count('!'),s.count(':'),s.count(';'),s.count('\n')]+[s.count(c)/den for c in PUNCT])
 return np.asarray(rows,dtype=np.float32)
word=TfidfVectorizer(analyzer='word',ngram_range=(1,3),min_df=2,sublinear_tf=True,max_features=120000)
char=TfidfVectorizer(analyzer='char',ngram_range=(1,5),min_df=2,sublinear_tf=True,max_features=120000)
Xw=word.fit_transform(text); Xc=char.fit_transform(text); sc=StandardScaler(); Xs=csr_matrix(sc.fit_transform(style(text))); X=hstack([Xw,Xc,Xs],format='csr')
print('shape',X.shape,'rows',len(y),'families',sorted(set(family)))


In [ ]:
if groups is not None:
 splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED); splits=list(splitter.split(X,y,groups))
else:
 splitter=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED); splits=list(splitter.split(X,y))

def fit_hierarchical(Xtr,ytr,ftr,Xv):
 fam_clf=LinearSVC(C=C,class_weight='balanced',max_iter=4000).fit(Xtr,ftr)
 fam_pred=fam_clf.predict(Xv)
 pred=np.empty(len(Xv),dtype=object)
 for fam in sorted(set(ftr)):
  tr=np.where(ftr==fam)[0]; va=np.where(fam_pred==fam)[0]
  labels=np.unique(ytr[tr])
  if len(va)==0: continue
  if len(labels)==1: pred[va]=labels[0]; continue
  local=LinearSVC(C=C,class_weight='balanced',max_iter=4000).fit(Xtr[tr],ytr[tr])
  pred[va]=local.predict(Xv[va])
 return pred, fam_pred

rows=[]
for fold,(tr,va) in enumerate(splits,1):
 direct=LinearSVC(C=C,class_weight='balanced',max_iter=4000).fit(X[tr],y[tr]).predict(X[va])
 hier,famp=fit_hierarchical(X[tr],y[tr],family[tr],X[va])
 direct_score=balanced_accuracy_score(y[va],direct); hier_score=balanced_accuracy_score(y[va],hier)
 fam_score=balanced_accuracy_score(family[va],famp)
 rows.append({'fold':fold,'direct':float(direct_score),'hierarchical':float(hier_score),'family_accuracy':float(fam_score)})
 print(f'fold={fold} direct={direct_score:.5f} hierarchical={hier_score:.5f} family={fam_score:.5f}')
summary={'experiment':'family_aware_tfidf','grouped_validation':groups is not None,'folds':rows,'direct_mean':float(np.mean([r['direct'] for r in rows])),'hierarchical_mean':float(np.mean([r['hierarchical'] for r in rows])),'family_mean':float(np.mean([r['family_accuracy'] for r in rows]))}
OUT.write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))


The JSON artifact reports direct 18-class balanced accuracy, hierarchical family-to-model accuracy, and family-level accuracy on identical folds.
